In [13]:
import os
from googleapiclient.discovery import build
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import difflib

# Налаштування API
API_KEY = "AI12zaSyCQ0d1E8wg2Hs1bWDL-Z5iH15t39kJ73UZ4fIw"
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Функція для збору трендових відео
def get_trending_videos(region_code='UA'):
    request = youtube.videos().list(
        part="snippet",
        chart="mostPopular",
        regionCode=region_code,
        maxResults=100
    )
    response = request.execute()
    return response.get('items', [])

# Функція для збору відео з каналу
def get_channel_videos(channel_id):
    videos = []
    next_page_token = None
    while True:
        response = youtube.search().list(
            channelId=channel_id,
            part='snippet',
            order='date',
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        videos.extend(response['items'])
        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
    return videos

# Функція для пошуку ключових слів
def check_videos_for_keyword(videos, keyword):
    matching_videos = []
    for video in videos:
        title = video['snippet'].get('title', '')
        description = video['snippet'].get('description', '')
        if keyword.lower() in title.lower() or keyword.lower() in description.lower():
            matching_videos.append(video)
    return matching_videos

# Функція для витягування video_id
def extract_video_id(video):
    if 'id' in video:
        if isinstance(video['id'], dict):
            return video['id'].get('videoId', 'unknown')
        return video['id']
    return 'unknown'

# Функція для збереження відео в базу даних
def save_to_database(videos, db_name="youtube_data.db", table_name="videos"):
    conn = sqlite3.connect(db_name)
    data = []
    for video in videos:
        video_id = extract_video_id(video)
        title = video['snippet'].get('title', '')
        description = video['snippet'].get('description', '')
        published_at = video['snippet'].get('publishedAt', '')
        data.append((video_id, title, description, published_at))
    
    df = pd.DataFrame(data, columns=['video_id', 'title', 'description', 'published_at'])
    df.to_sql(table_name, conn, if_exists='append', index=False)
    conn.close()
    print(f"Дані збережено в {db_name}, таблиця {table_name}")

# Функція для аналізу тригерних слів (аномалій)
def analyze_triggers(db_name="youtube_data.db", table_name="videos"):
    trigger_words = ["шок", "скандал", "терміново", "сенсація", "вина", "трагедія"]
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    df['trigger_count'] = df['title'].apply(lambda x: sum(word.lower() in x.lower() for word in trigger_words)) + \
                         df['description'].apply(lambda x: sum(word.lower() in x.lower() for word in trigger_words))
    
    anomalies = df[df['trigger_count'] > 2]  # Аномалія: більше 2 тригерних слів
    print(f"Знайдено аномалій: {len(anomalies)}")
    return anomalies

# Функція для виявлення ботів за частотою публікацій
def detect_bots_by_frequency(db_name="youtube_data.db", table_name="videos"):
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['date'] = df['published_at'].dt.date
    video_counts = df.groupby('date').size()
    
    suspicious_days = video_counts[video_counts > 10]  # Підозріло: більше 10 відео на день
    print(f"Підозрілі дні з великою кількістю відео: {len(suspicious_days)}")
    return suspicious_days

# Функція для виявлення ботів за схожістю заголовків
def detect_bots_by_similar_titles(db_name="youtube_data.db", table_name="videos"):
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    titles = df['title'].tolist()
    similar_titles = []
    
    for i in range(len(titles)):
        for j in range(i + 1, len(titles)):
            similarity = difflib.SequenceMatcher(None, titles[i], titles[j]).ratio()
            if similarity > 0.9:  # Якщо схожість > 90%
                similar_titles.append((titles[i], titles[j], similarity))
    
    print(f"Знайдено схожих заголовків: {len(similar_titles)}")
    return similar_titles

# Оновлена функція main
def main(get_videos_func, func_arg, keyword, table_name):
    print(f"\n Пошук відео для ключового слова: '{keyword}'\n")
    video_info = get_videos_func(func_arg)

    print(f"Знайдено {len(video_info)} відео. Виводжу всі назви:\n")
    for video in video_info:
        print("🎬", video['snippet'].get('title', ''))

    # Збереження в базу даних
    save_to_database(video_info, table_name=table_name)

    # Пошук ключового слова
    matching_videos = check_videos_for_keyword(video_info, keyword)
    print(f"\n✅ Виявлено {len(matching_videos)} відео з ключовим словом '{keyword}':\n")
    for video in matching_videos:
        title = video['snippet'].get('title', '')
        date = video['snippet'].get('publishedAt', '---')
        video_id = extract_video_id(video)
        url = f"https://www.youtube.com/watch?v={video_id}"
        print(f"📌 Title: {title}\n🔗 URL: {url}\n📅 Date: {date}\n")

    # Аналіз аномалій
    anomalies = analyze_triggers(table_name=table_name)
    print("Аномалії (тригерні слова):")
    print(anomalies[['title', 'trigger_count']])

    # Виявлення ботів за частотою
    suspicious_days = detect_bots_by_frequency(table_name=table_name)
    print("Підозрілі дні (можливі боти):")
    print(suspicious_days)

    # Виявлення ботів за схожістю заголовків
    similar_titles = detect_bots_by_similar_titles(table_name=table_name)
    print("Схожі заголовки (можливі боти):")
    for title1, title2, similarity in similar_titles[:5]:  # Виводимо перші 5 пар
        print(f"Схожі заголовки: {title1} | {title2} | Схожість: {similarity:.2f}")

In [14]:
main(get_trending_videos, 'UA', 'війна', 'trending_videos')


 Пошук відео для ключового слова: 'війна'

Знайдено 50 відео. Виводжу всі назви:

🎬 У дім Остапа Курляка влучила російська ракета. Його родичі живуть у Хабаровську і вірять пропаганді
🎬 «Ми безшумно заїжджали і розстрілювали»: бойові роботи ЗСУ
🎬 Мега стройки палками - фейк #амитеш
🎬 "Хоч щось, але забираємо". Санітар медичного підрозділу "Форсаж" 56 ОМПБр про евакуацію з фронту
🎬 В ДЕТСТВЕ ЗАБЫЛ ПАРОЛЬ ОТ ШАЛАША
🎬 Жителі Запорізької області діляться думками про імовірний наступ РФ
🎬 😯 Хотел сделать фото с Бугатти за €4.000.000, но реакция хозяйки удивила! | Новостничок
🎬 Что делает гвардеец когда до него докапываются? #амитеш
🎬 ГДЕ НАСТОЯЩАЯ КВИНКА😱👑💖🍍Смотри до конца и узнаешь,нашли или нет😂#роблокс #игры #смешное #квинка
🎬 Smart Sigma Kid #funny #sigma
🎬 Она сделает ваш день 🔥
🎬 Вот КАК МИДИИ Мониторят КАЧЕСТВО Воды в ГОРОДЕ! #шортс
🎬 sorry🥲 @andrey.grechka (fake bruises)
🎬 ⚡️НИКТО даже ПРЕДСТАВИТЬ не мог, что её можно ОСТАНОВИТЬ...⚡️ #shorts
🎬 ПРЕМЬЕРА! НОВИНКА 2025! Собиратель кам

In [ ]:
main(get_channel_videos, 'UC7AvvBtzbwWpkcwxV4bGKLw', 'подорож', 'channel_videos')


 Пошук відео для ключового слова: 'подорож'

Знайдено 89 відео. Виводжу всі назви:

🎬 Головний скарб Норвегії, що прихований за полярним колом.
🎬 Нащо йдуть на Еверест? Гірська хвороба, наука, побут та комерція на екстремальних висотах.
🎬 Дивовижне життя у дивовижних Гімалаях.
🎬 Ті самі дні на Драгобраті
🎬 Цю країну не зрозуміти. НЕПАЛ, Слони в кайданах, українські літаки та їжа.
🎬 Хаос, який шокує. НЕПАЛ, Катманду, мавпи, мотоцикли та корупція.
🎬 Чому я став бігати горами замість походів з наметом?
🎬 Раменоманія, ідеальна яловичина, небезпечна риба та огидний суперпродукт. ЯПОНІЯ та її їжа.
🎬 Справжня ЯПОНІЯ. Таємниці Хірошими, гейші Кіото,  безлюдні гори, найшвидші потяги та ніч у храмі.
🎬 Україна і Японія. Життя, освіта, спорт і чому вони нам допомагають.
🎬 Найдивніший мегаполіс світу та як ЯПОНІЯ стала такою, як вона є.
🎬 Зимова ЯПОНІЯ, про яку ви не знали, найкращий у світі сніг, острів, що ледь не став СРСР.
🎬 Більш детальний тест- драйв невдовзі на каналі @TERENUA
🎬 Не шукайте 